strategie basé sur les données de jpbox 

ajout de la colonne box_office_us     box_office_fr   duration    languages  audience   budget  

In [1]:
import pandas as pd

# Charger le fichier final_clustered.csv
final_df = pd.read_csv('final_clustered.csv')

# Charger le fichier xgboost.csv
xgb_df = pd.read_csv('xgboost.csv')

# Sélectionner uniquement les colonnes à ajouter depuis xgboost.csv
cols_to_keep = ['title', 'box_office_fr', 'box_office_us', 'duration', 'languages', 'audience', 'budget']
xgb_subset = xgb_df[cols_to_keep]

# Fusionner final_df avec xgb_subset en utilisant une jointure à gauche (left join)
# Nous utilisons 'titre' de final_df et 'title' de xgb_subset comme clés de jointure.
merged_df = final_df.merge(xgb_subset, how='left', left_on='titre', right_on='title')

# Supprimer la colonne 'title' qui est redondante avec 'titre'
merged_df.drop(columns=['title'], inplace=True)

# Sauvegarder le résultat dans un nouveau fichier (facultatif)
merged_df.to_csv('modele3.csv', index=False)

# Afficher quelques lignes pour vérification
print(merged_df.head())


   film_id  realisateur_id rang  \
0    13345           575.0    3   
1      701           147.0    7   
2    17528           505.0    8   
3     2398           438.0    5   
4    10845           333.0    9   

                                               titre  \
0                   Star Wars: Le Réveil de la Force   
1                               La Revanche des Sith   
2                                 Le Roi Lion (2019)   
3                                             Taxi 2   
4  Les Aventures de Tintin : Le secret de la Licorne   

                                          titre_vo       realisateur  \
0                     Star Wars: The Force Awakens       J.J. Abrams   
1                              Revenge of the Sith      George Lucas   
2                             The Lion King (2019)       Jon Favreau   
3                                           Taxi 2   Gérard Krawczyk   
4  The Adventures of Tintin: Secret of the Unicorn  Steven Spielberg   

               genr

ajout de la colonne voix off

In [2]:
import pandas as pd
import ast

def extract_voix_off_names(acteurs_str):
    """
    Extrait les noms complets des acteurs dont le rôle est 'Voix-off'
    à partir d'une chaîne de caractères représentant une liste de dictionnaires.
    La fonction retourne une chaîne de noms séparés par des virgules.
    """
    try:
        # Convertir la chaîne en liste de dictionnaires
        acteurs_list = ast.literal_eval(acteurs_str)
    except Exception as e:
        # En cas d'erreur de parsing, retourner une chaîne vide
        return ""
    
    noms = []
    for acteur in acteurs_list:
        # Vérifier que l'entrée est bien un dictionnaire
        if isinstance(acteur, dict):
            # Comparer le rôle en ignorant la casse
            if acteur.get('role', '').lower() == 'voix-off':
                # Récupérer le nom complet tel qu'il est présent dans le dictionnaire
                full_name = acteur.get('name', '').strip()
                if full_name:
                    noms.append(full_name)
    # Joindre les noms avec une virgule pour créer la chaîne finale
    return ", ".join(noms)

# Charger le fichier modele3.csv
df = pd.read_csv("modele3.csv")

# Appliquer la fonction d'extraction sur la colonne 'acteurs' pour créer la nouvelle colonne 'voix_off'
df["voix_off"] = df["acteurs"].apply(extract_voix_off_names)

# Sauvegarder le résultat dans un nouveau fichier CSV
df.to_csv("modele3voixoff.csv", index=False)

# Afficher quelques lignes pour vérifier le résultat
print(df.head())


   film_id  realisateur_id rang  \
0    13345           575.0    3   
1      701           147.0    7   
2    17528           505.0    8   
3     2398           438.0    5   
4    10845           333.0    9   

                                               titre  \
0                   Star Wars: Le Réveil de la Force   
1                               La Revanche des Sith   
2                                 Le Roi Lion (2019)   
3                                             Taxi 2   
4  Les Aventures de Tintin : Le secret de la Licorne   

                                          titre_vo       realisateur  \
0                     Star Wars: The Force Awakens       J.J. Abrams   
1                              Revenge of the Sith      George Lucas   
2                             The Lion King (2019)       Jon Favreau   
3                                           Taxi 2   Gérard Krawczyk   
4  The Adventures of Tintin: Secret of the Unicorn  Steven Spielberg   

               genr

ajout des voix off en dans le cluster acteur principal

In [4]:
import pandas as pd
import ast

# Charger le fichier modele3.csv dans un DataFrame
df = pd.read_csv("modele3acteurclustercorrige.csv")

# Si la colonne par erreur 'cluster_acteur_principale' existe, on la supprime.
if 'cluster_acteur_principale' in df.columns:
    df.drop(columns=['cluster_acteur_principale'], inplace=True)

# Charger le fichier actors.csv qui contient les correspondances
actors_df = pd.read_csv("actors.csv")

def get_cluster_from_voix_off(voix_off_str):
    """
    Pour une chaîne contenant les noms séparés par des virgules provenant de la colonne 'voix_off',
    recherche dans actors_df une correspondance exacte dans la colonne 'acteur'.
    Retourne la valeur de la colonne 'cluster_acteur' du premier acteur trouvé.
    Si aucune correspondance n'est trouvée, retourne une chaîne vide.
    """
    if pd.isna(voix_off_str) or voix_off_str.strip() == "":
        return ""
    
    # Découper la chaîne en liste de noms en supprimant les espaces superflus
    acteurs_list = [nom.strip() for nom in voix_off_str.split(",")]
    for acteur in acteurs_list:
        matching = actors_df[actors_df['acteur'] == acteur]
        if not matching.empty:
            return matching.iloc[0]['cluster_acteur']
    return ""

def update_cluster_acteur_principal(row):
    """
    Si la colonne 'cluster_acteur_principal' est vide ou manquante pour la ligne,
    la remplit avec le cluster obtenu via la fonction get_cluster_from_voix_off appliquée sur 'voix_off'.
    Sinon, conserve la valeur existante.
    """
    current_value = row.get("cluster_acteur_principal", "")
    if pd.isna(current_value) or str(current_value).strip() == "":
        return get_cluster_from_voix_off(row["voix_off"])
    else:
        return current_value

# Met à jour la colonne 'cluster_acteur_principal' en appliquant la fonction sur chaque ligne
df["cluster_acteur_principal"] = df.apply(update_cluster_acteur_principal, axis=1)

# Sauvegarder le DataFrame mis à jour dans un nouveau fichier CSV
df.to_csv("modele3.csv", index=False)

# Afficher quelques lignes pour vérification
print(df[["voix_off", "cluster_acteur_principal"]].head())



                                voix_off cluster_acteur_principal
0                                    NaN                        0
1                                    NaN                        1
2  Jamel Debbouze, Jean Reno, Seth Rogen                        2
3                                    NaN                        0
4  Daniel Craig, Gad Elmaleh, Simon Pegg                        0


ajout de la colonne demarrage

In [5]:
import pandas as pd

# Charger le fichier modele3.csv
modele_df = pd.read_csv("modele3.csv")

# Charger le fichier JPboxfusionmaxelie.csv
jpbox_df = pd.read_csv("JPboxfusionmaxelie.csv")

# Sélectionner les colonnes d'intérêt dans le fichier JPboxfusionmaxelie.csv
# Ici, on garde "Titre" (clé de jonction) et "Démarrage" que l'on souhaite ajouter
jpbox_subset = jpbox_df[['Titre', 'Démarrage']]

# Fusionner les DataFrames par une jointure à gauche (modele3 ne doit pas voir ses lignes augmentées)
merged_df = modele_df.merge(jpbox_subset, how='left', left_on='titre', right_on='Titre')

# Renommer la colonne "Démarrage" en "demarrage"
merged_df.rename(columns={'Démarrage': 'demarrage'}, inplace=True)

# Supprimer la colonne "Titre" importée pour éviter la redondance (optionnel)
merged_df.drop(columns=['Titre'], inplace=True)

# Sauvegarder le DataFrame mis à jour
merged_df.to_csv("modele3_updated.csv", index=False)

# Afficher quelques lignes pour vérification
print(merged_df.head())


   film_id  realisateur_id rang  \
0    13345           575.0    3   
1      701           147.0    7   
2    17528           505.0    8   
3     2398           438.0    5   
4    10845           333.0    9   

                                               titre  \
0                   Star Wars: Le Réveil de la Force   
1                               La Revanche des Sith   
2                                 Le Roi Lion (2019)   
3                                             Taxi 2   
4  Les Aventures de Tintin : Le secret de la Licorne   

                                          titre_vo       realisateur  \
0                     Star Wars: The Force Awakens       J.J. Abrams   
1                              Revenge of the Sith      George Lucas   
2                             The Lion King (2019)       Jon Favreau   
3                                           Taxi 2   Gérard Krawczyk   
4  The Adventures of Tintin: Secret of the Unicorn  Steven Spielberg   

               genr

In [6]:
import pandas as pd

# Charger le fichier modele3.csv
modele_df = pd.read_csv("modele3.csv")

# Charger le fichier fusioneli.csv
fusioneli_df = pd.read_csv("fusioneli.csv")

# Sélectionner uniquement les colonnes 'Titre' et 'Démarrage' du fichier fusioneli.csv
fusioneli_subset = fusioneli_df[['Titre', 'Démarrage']]

# Fusionner les DataFrames en effectuant une jointure à gauche :
# - 'titre' du DataFrame modele_df correspond à 'Titre' du DataFrame fusioneli_subset.
merged_df = modele_df.merge(fusioneli_subset, how='left', left_on='titre', right_on='Titre')

# Supprimer la colonne 'Titre' importée depuis fusioneli.csv pour éviter la redondance
merged_df.drop(columns=['Titre'], inplace=True)

# Optionnel : renommer la colonne 'Démarrage' en 'demarrage' pour une cohérence éventuelle dans le DataFrame
merged_df.rename(columns={'Démarrage':'demarrage'}, inplace=True)

# Sauvegarder le DataFrame mis à jour dans un nouveau fichier CSV
merged_df.to_csv("modele3_updated.csv", index=False)

# Afficher un aperçu pour vérification
print(merged_df.head())


   film_id  realisateur_id rang  \
0    13345           575.0    3   
1      701           147.0    7   
2    17528           505.0    8   
3     2398           438.0    5   
4    10845           333.0    9   

                                               titre  \
0                   Star Wars: Le Réveil de la Force   
1                               La Revanche des Sith   
2                                 Le Roi Lion (2019)   
3                                             Taxi 2   
4  Les Aventures de Tintin : Le secret de la Licorne   

                                          titre_vo       realisateur  \
0                     Star Wars: The Force Awakens       J.J. Abrams   
1                              Revenge of the Sith      George Lucas   
2                             The Lion King (2019)       Jon Favreau   
3                                           Taxi 2   Gérard Krawczyk   
4  The Adventures of Tintin: Secret of the Unicorn  Steven Spielberg   

               genr

In [7]:
import pandas as pd
import numpy as np

# Charger le fichier modele3_updated.csv
df = pd.read_csv("modele3_updated.csv")

# Remplacer les chaînes vides par des NaN dans la colonne 'demarrage'
df['demarrage'] = df['demarrage'].replace(r'^\s*$', np.nan, regex=True)

# Pour les lignes où 'demarrage' est manquant, utiliser la valeur de 'box_office_fr'
df['demarrage'] = df['demarrage'].fillna(df['box_office_fr'])

# Sauvegarder le DataFrame mis à jour
df.to_csv("modele3_updated.csv", index=False)

# Vérifier les résultats
print(df[['box_office_fr', 'demarrage']].head())


   box_office_fr  demarrage
0            NaN        NaN
1            NaN        NaN
2            NaN        NaN
3            NaN        NaN
4            NaN        NaN


In [8]:
import pandas as pd

# Charger le fichier modele3_updated.csv
modele_df = pd.read_csv("modele3_updated.csv")

# Charger le fichier leo.csv
leo_df = pd.read_csv("leo.csv")

# Sélectionner uniquement les colonnes nécessaires dans leo.csv
leo_subset = leo_df[['titre', 'box_office_demarrage']]

# Fusionner les deux fichiers par une jointure à gauche
# La colonne 'titre' est la clé de jointure dans les deux fichiers
merged_df = modele_df.merge(leo_subset, how='left', on='titre')

# Renommer la colonne 'box_office_demarrage' en 'boxofficeleo'
merged_df.rename(columns={'box_office_demarrage': 'boxofficeleo'}, inplace=True)

# Sauvegarder le fichier mis à jour
merged_df.to_csv("modele3_leo.csv", index=False)

# Afficher un aperçu pour vérification
print(merged_df.head())


   film_id  realisateur_id rang  \
0    13345           575.0    3   
1      701           147.0    7   
2    17528           505.0    8   
3     2398           438.0    5   
4    10845           333.0    9   

                                               titre  \
0                   Star Wars: Le Réveil de la Force   
1                               La Revanche des Sith   
2                                 Le Roi Lion (2019)   
3                                             Taxi 2   
4  Les Aventures de Tintin : Le secret de la Licorne   

                                          titre_vo       realisateur  \
0                     Star Wars: The Force Awakens       J.J. Abrams   
1                              Revenge of the Sith      George Lucas   
2                             The Lion King (2019)       Jon Favreau   
3                                           Taxi 2   Gérard Krawczyk   
4  The Adventures of Tintin: Secret of the Unicorn  Steven Spielberg   

               genr

In [10]:
import pandas as pd
import numpy as np

# Charger le fichier modele3_updated_final.csv
df = pd.read_csv("modele3_leo.csv")

# Créer une nouvelle colonne fusionnée en suivant les règles
# Si 'demarrage' et 'boxofficeleo' contiennent des chiffres, on conserve 'demarrage'.
# Sinon, on utilise la valeur existante (dans l'une des colonnes ou NaN).
df['demarrage_final'] = np.where(
    (~df['demarrage'].isna()) & (~df['boxofficeleo'].isna()),  # Les deux colonnes contiennent des chiffres
    df['demarrage'],  # Priorité à 'demarrage'
    df['demarrage'].combine_first(df['boxofficeleo'])  # Sinon, on fusionne en conservant une valeur si possible
)

# Supprimer les colonnes d'origine si elles ne sont plus nécessaires (optionnel)
df.drop(columns=['demarrage', 'boxofficeleo'], inplace=True)

# Sauvegarder le fichier mis à jour
df.to_csv("modele3_updated.csv", index=False)

# Afficher un aperçu des résultats
print(df.head())


   film_id  realisateur_id rang  \
0    13345           575.0    3   
1      701           147.0    7   
2    17528           505.0    8   
3     2398           438.0    5   
4    10845           333.0    9   

                                               titre  \
0                   Star Wars: Le Réveil de la Force   
1                               La Revanche des Sith   
2                                 Le Roi Lion (2019)   
3                                             Taxi 2   
4  Les Aventures de Tintin : Le secret de la Licorne   

                                          titre_vo       realisateur  \
0                     Star Wars: The Force Awakens       J.J. Abrams   
1                              Revenge of the Sith      George Lucas   
2                             The Lion King (2019)       Jon Favreau   
3                                           Taxi 2   Gérard Krawczyk   
4  The Adventures of Tintin: Secret of the Unicorn  Steven Spielberg   

               genr

In [11]:
import pandas as pd

# Charger le fichier modele3_updated_final_with_demarrage.csv
df = pd.read_csv("modele3_updated.csv")

# Vérifier les lignes où 'demarrage_final' est manquant ou vide
missing_count = df['demarrage_final'].isna().sum()  # Compter les NaN (valeurs manquantes)
empty_count = (df['demarrage_final'] == "").sum()  # Compter les chaînes vides

# Ajouter les deux cas pour obtenir le total des lignes sans information
total_missing_or_empty = missing_count + empty_count

# Afficher le résultat
print(f"Nombre de lignes sans information dans 'demarrage_final' : {total_missing_or_empty}")


Nombre de lignes sans information dans 'demarrage_final' : 1479


In [12]:
import pandas as pd

# Charger le fichier modele3_updated_final_with_demarrage.csv
df = pd.read_csv("modele3_updated.csv")

# Identifier et supprimer les lignes sans information dans 'demarrage_final'
df = df[~df['demarrage_final'].isna()]  # Supprimer les NaN
df = df[df['demarrage_final'] != ""]    # Supprimer les chaînes vides

# Sauvegarder le DataFrame mis à jour dans un nouveau fichier CSV
df.to_csv("modele3_cleaned.csv", index=False)

# Afficher un aperçu pour vérifier les changements
print(f"Nombre de lignes restantes : {len(df)}")
print(df.head())


Nombre de lignes restantes : 7395
   film_id  realisateur_id rang  \
0    13345           575.0    3   
1      701           147.0    7   
2    17528           505.0    8   
3     2398           438.0    5   
4    10845           333.0    9   

                                               titre  \
0                   Star Wars: Le Réveil de la Force   
1                               La Revanche des Sith   
2                                 Le Roi Lion (2019)   
3                                             Taxi 2   
4  Les Aventures de Tintin : Le secret de la Licorne   

                                          titre_vo       realisateur  \
0                     Star Wars: The Force Awakens       J.J. Abrams   
1                              Revenge of the Sith      George Lucas   
2                             The Lion King (2019)       Jon Favreau   
3                                           Taxi 2   Gérard Krawczyk   
4  The Adventures of Tintin: Secret of the Unicorn  Steven

In [13]:
import pandas as pd

# Charger le fichier CSV
df = pd.read_csv("modele3_cleaned.csv")  # Remplacez par le nom de votre fichier

# Considérons que les colonnes s'appellent "pays" (pour les codes, ex. "f23") 
# et "Pays" (pour les noms, ex. "france").

# 1. Construire les mappings à partir des lignes où les deux informations sont présentes
# On s'assure que les deux colonnes ne sont pas NaN
df_clean = df.dropna(subset=["pays", "Pays"])

# Construction du mapping code → nom (par exemple, "f23" -> "usa") et
# mapping nom → code (par exemple, "usa" -> "f23")
mapping_code_to_country = df_clean.drop_duplicates(subset=["pays"]).set_index("pays")["Pays"].to_dict()
mapping_country_to_code = df_clean.drop_duplicates(subset=["Pays"]).set_index("Pays")["pays"].to_dict()

# 2. Créer des fonctions pour compléter les colonnes manquantes

def fill_code(row):
    """
    Si la colonne "pays" (code) est manquante mais que "Pays" est renseigné,
    on tente de récupérer le code correspondant grâce au mapping.
    """
    if pd.isna(row["pays"]) and pd.notna(row["Pays"]):
        return mapping_country_to_code.get(row["Pays"], row["pays"])
    else:
        return row["pays"]

def fill_country(row):
    """
    Si la colonne "Pays" (nom) est manquante mais que "pays" est renseigné,
    on complète la valeur en utilisant le mapping code → nom.
    """
    if pd.isna(row["Pays"]) and pd.notna(row["pays"]):
        return mapping_code_to_country.get(row["pays"], row["Pays"])
    else:
        return row["Pays"]

# 3. Appliquer ces fonctions sur le DataFrame pour mettre à jour les colonnes
df["pays"] = df.apply(fill_code, axis=1)
df["Pays"] = df.apply(fill_country, axis=1)

# Sauvegarder le résultat dans un nouveau fichier CSV, si besoin
df.to_csv("modele3.csv", index=False)

print("Mise à jour terminée.")


Mise à jour terminée.


In [16]:
import pandas as pd
import unicodedata
import re

# Fonction de nettoyage des titres pour normalisation
def clean_title(title: str) -> str:
    if pd.isnull(title):
        return title
    t = str(title).lower().strip()
    t = unicodedata.normalize('NFKD', t).encode('ascii', 'ignore').decode('ascii')
    t = re.sub(r'[^a-z0-9 ]', '', t)
    t = re.sub(r'\s+', ' ', t)
    return t

# Lecture des deux fichiers CSV
modele3 = pd.read_csv('modele3.csv')
allocine = pd.read_csv('allocinejpboxv4.csv')

# Création de la clé de jointure normalisée
modele3['title_clean'] = modele3['title'].apply(clean_title)
allocine['title_clean'] = allocine['title'].apply(clean_title)

# Sélection et agrégation des colonnes utiles de allocine
to_merge = allocine[['title_clean', 'duration', 'writer', 'languages']]
to_merge_agg = (
    to_merge
    .groupby('title_clean', as_index=False)
    .agg({
        'duration': 'first',
        'writer': lambda x: ';'.join(x.dropna().unique()),
        'languages': lambda x: ';'.join(x.dropna().unique())
    })
)

# Jointure gauche pour enrichir modele3
merged = pd.merge(
    modele3,
    to_merge_agg,
    on='title_clean',
    how='left'
)

# Fusion des colonnes languages_x et languages_y en ne conservant que la première langue
# languages_x : éventuelle colonne languages d'origine (modele3), languages_y : de allocine

def first_language(x, y):
    # Choisir x si existant sinon y, puis ne garder que la première langue avant ';'
    val = x if pd.notnull(x) and str(x).strip() else y
    if pd.isnull(val):
        return None
    return str(val).split(';')[0]

merged['languages'] = merged.apply(
    lambda row: first_language(row.get('languages_x'), row.get('languages_y')),
    axis=1
)

# Suppression des colonnes temporaires
cols_to_drop = ['title_clean', 'languages_x', 'languages_y']
merged.drop(columns=[c for c in cols_to_drop if c in merged.columns], inplace=True)

# Sauvegarde du résultat
merged.to_csv('modele3_enriched.csv', index=False)
print("Fichier 'modele3_enriched.csv' généré avec la colonne 'languages' fusionnée (première langue conservée).")



Fichier 'modele3_enriched.csv' généré avec la colonne 'languages' fusionnée (première langue conservée).


In [20]:
import pandas as pd
import ast

# Lecture du fichier CSV initial
df = pd.read_csv('modele3.csv')

# Fonction de conversion qui gère les valeurs manquantes en renvoyant une liste vide
def parse_actors(val):
    if pd.isnull(val):
        return []
    return ast.literal_eval(val)

# Conversion de la colonne "moyennes_individuelles_acteurs"
df["moyennes_individuelles_acteurs"] = df["moyennes_individuelles_acteurs"].apply(parse_actors)

# Calcul de la somme des moyennes pour chaque ligne. Si la liste est vide, la somme sera 0.
df["sommedesmoyennes"] = df["moyennes_individuelles_acteurs"].apply(
    lambda acteurs: sum(actor["moyenne"] for actor in acteurs)
)

# Enregistrement du DataFrame modifié dans un nouveau fichier CSV
df.to_csv('modele3v2.csv', index=False)

print("Le fichier a été enregistré dans modele3v2.csv")


Le fichier a été enregistré dans modele3v2.csv


In [21]:
import pandas as pd

# Lecture du fichier CSV prévu
df = pd.read_csv('modele3v2.csv')

# Fusion des colonnes duration_x et duration_y dans une nouvelle colonne 'duration'
# Ici, on utilise fillna pour utiliser la valeur de duration_y lorsque duration_x est manquante.
df['duration'] = df['duration_x'].fillna(df['duration_y'])

# Si vous souhaitez utiliser combine_first (équivalent) :
# df['duration'] = df['duration_x'].combine_first(df['duration_y'])

# Éventuellement, vous pouvez supprimer duration_x et duration_y si vous n'en avez plus besoin
df.drop(['duration_x', 'duration_y'], axis=1, inplace=True)

# Enregistrement du DataFrame dans un nouveau fichier CSV
df.to_csv('modele3v3.csv', index=False)

print("Le fichier mis à jour a été enregistré dans modele3v3.csv")


Le fichier mis à jour a été enregistré dans modele3v3.csv


In [22]:
import pandas as pd

# Lecture du fichier CSV contenant déjà la colonne 'duration'
df = pd.read_csv('modele3v3.csv')

# Conversion de 'duration' en numérique. 
# Les valeurs erronées ou vides seront converties en NaN.
df['duration'] = pd.to_numeric(df['duration'], errors='coerce')

# Calcul de la médiane de la colonne 'duration' (en ignorant les NaN)
median_duration = df['duration'].median()

# Remplissage des valeurs manquantes (NaN) avec la médiane calculée
df['duration'] = df['duration'].fillna(median_duration)

# Enregistrement du DataFrame modifié dans un nouveau CSV
df.to_csv('modele3v4.csv', index=False)

print("Le fichier a été enregistré dans modele3v4.csv")


Le fichier a été enregistré dans modele3v4.csv


In [23]:
import pandas as pd

# Lecture du fichier CSV
df = pd.read_csv('modele3v4.csv')

# Remplissage des valeurs manquantes dans 'cluster_realisateur' par 0
df['cluster_realisateur'] = df['cluster_realisateur'].fillna(0)

# Conversion de la colonne en type float puis en int (dans le cas où les valeurs seraient des floats)
df['cluster_realisateur'] = df['cluster_realisateur'].astype(float).astype(int)

# Enregistrement du DataFrame dans un nouveau fichier CSV
df.to_csv('modele3v5.csv', index=False)

print("Le fichier mis à jour a été enregistré dans modele3v5.csv")


Le fichier mis à jour a été enregistré dans modele3v5.csv


In [25]:
import pandas as pd

def invert_clusters(x):
    """
    Inverse 1 et 0 dans une cellule.
    Si la cellule contient plusieurs chiffres séparés par une virgule,
    chaque élément est traité individuellement.
    Si une valeur n'est pas 0 ou 1, elle reste inchangée.
    """
    if pd.isnull(x):
        return x  # Ne rien changer si NaN

    # Convertir la valeur en chaîne et supprimer d'éventuels crochets
    x_str = str(x).replace('[','').replace(']','').strip()

    # Si la cellule contient une virgule, on suppose qu'il y a plusieurs chiffres
    if "," in x_str:
        # Séparer les éléments et nettoyer les espaces
        parts = [part.strip() for part in x_str.split(",")]
        inverted_parts = []
        for part in parts:
            if part == "1":
                inverted_parts.append("0")
            elif part == "0":
                inverted_parts.append("1")
            else:
                # Conserver la valeur si ce n'est pas 0 ou 1
                inverted_parts.append(part)
        # On retourne une chaîne avec les valeurs inversées, séparées par ", "
        return ", ".join(inverted_parts)
    else:
        # Cas d'une seule valeur
        if x_str == "1":
            return 0
        elif x_str == "0":
            return 1
        else:
            # Tenter de convertir en entier pour voir si c'est 0 ou 1, sinon laisser
            try:
                num = int(x_str)
                if num == 1:
                    return 0
                elif num == 0:
                    return 1
                else:
                    return num
            except:
                return x_str

# Lecture du fichier CSV existant (par exemple modele3v_final.csv)
df = pd.read_csv('modele3v5.csv')

# Appliquer la transformation sur chaque colonne cible
df['cluster_acteur_principal'] = df['cluster_acteur_principal'].apply(invert_clusters)
df['cluster_acteur_secondaire'] = df['cluster_acteur_secondaire'].apply(invert_clusters)

# Sauvegarder le DataFrame modifié dans un nouveau fichier CSV
df.to_csv('modele3v6.csv', index=False)

print("Les colonnes 'cluster_acteur_principal' et 'cluster_acteur_secondaire' ont été inversées et le résultat est enregistré dans modele3v.csv")



Les colonnes 'cluster_acteur_principal' et 'cluster_acteur_secondaire' ont été inversées et le résultat est enregistré dans modele3v.csv


In [26]:
import pandas as pd

# Lecture du fichier CSV (ajustez le nom du fichier si nécessaire)
df = pd.read_csv('modele3v6.csv')

# Remplissage des valeurs manquantes (NaN) par 0 dans les colonnes concernées
df['cluster_acteur_principal'] = df['cluster_acteur_principal'].fillna(0)
df['cluster_acteur_secondaire'] = df['cluster_acteur_secondaire'].fillna(0)

# Remplacement des chaînes vides (éventuelles) par 0
df['cluster_acteur_principal'] = df['cluster_acteur_principal'].replace(r'^\s*$', 0, regex=True)
df['cluster_acteur_secondaire'] = df['cluster_acteur_secondaire'].replace(r'^\s*$', 0, regex=True)

# Sauvegarde du DataFrame modifié dans un nouveau fichier CSV
df.to_csv('modele3v7.csv', index=False)

print("Les entrées vides dans 'cluster_acteur_principal' et 'cluster_acteur_secondaire' ont été remplacées par 0.")


Les entrées vides dans 'cluster_acteur_principal' et 'cluster_acteur_secondaire' ont été remplacées par 0.
